# PreProcessing BIDS-Style MRI Images with Multi-Modal Brain Scans
This noteboook prepares a post-operative BIDS-Style MRI dataset for training of an Unsupervised/Self-Supervised Learning (SSL) model.  The BIDS-like structure consists of NIfTI MRI files for the following modalities:
- **T1c (T1-weighted Contrast-enhanced**:  T1c weighted images are taken after a contrasting agent has been injected into the patient to identify areas where the blood-brain barrier is broken, such as active tumor margins
- **T1n (T1-weighted / Native)**: Structural scans that highlight anatomical details, where fat appears bright and fluid appears dark
- **T2f (FLAIR-Fluid Attenuated Inversion Recovery)**: This a T2-weighted sequence that suppresses the signal from free water (like cerebrospinal fluid), making it ideal for detecting peritumoral edema or lesions (bright) without signal interference from nearby fluid
- **T2w (T2-weighted)**: Sequence that highlight water and fat, making them bright, which is highly sensitive for identifying inflammation, edema, and tumors

----
<a name='configure_environment'></a>
## 1.0 <span style='color:blue'>|</span> Setup Environment

In [1]:
import os, sys, logging, glob
from pathlib import Path
from typing import List, Dict, Tuple
from functools import partial

import nibabel as nib
import numpy as np
from nilearn.image import resample_img
from joblib import Parallel, delayed

In [2]:
# Define Global Configuration
TARGET_SHAPE = (128, 128, 64) 
TARGET_AFFINE = np.diag([2.0, 2.0, 3.0, 1.0])

# Modalities to resample (excluding tumor mask, handled separately)
MODALITIES = ['t1n', 't1c', 't2w', 't2f']
MASK_MODALITY = 'tumorMask'

# Input / output paths
ROOT_DIR = Path('../post')  
OUTPUT_ROOT = Path('../post/output') / 'preproc'

# Parallel processing
N_JOBS = 16  # Adjust based on CPU cores

In [3]:
def get_patient_dirs(root: Path) -> List[Path]:
    '''
    Find all patient directories (e.g., 'PatientID_*') in BIDS root.
    '''

    return sorted([d for d in root.iterdir() if d.is_dir() and d.name.startswith('PatientID')])

In [5]:
# Get all of the NIfTI file paths
def get_NIfTI_path(root: str, modality: str):
    '''
    Find NIfTI (.nii.gz) files matching a given modality (e.g. 't1c', 't1n').

    Args:
        root (str): Root directory path containing PatientID*/Timepoint* files.
        modality (str): Modality substring (e.g. 't1c'). Case-insensitive

    Returns:
        list[str]: Absolute path to the matching .nii.gz files
    '''
    
    root = Path(root)
    modality_lower = modality.lower()

    results = []

    for patient_dir in root.iterdir():
        if patient_dir.is_dir():
            for timepoint_dir in patient_dir.iterdir():
                for file in timepoint_dir.glob('*.nii.gz'):
                    if modality_lower in file.name.lower():
                        results.append(str(file.resolve()))
    return results

In [6]:
def resample_image(img_path: Path, target_affine: np.ndarray,
                   target_shape: Tuple[int, int, int],
                   interpolation: str = 'continuous', **kwargs) -> nib.Nifti1Image:
    '''
    Resample a NIfTI image to target affine/shape.
    Uses nilearn's resample_img for reliability
    '''
    
    if not img_path.exists():
        raise FileNotFoundError(f'File not found: {img_path}')

    img = nib.load(img_path)
    resampled = resample_img(img, target_affine=target_affine,
                             target_shape=target_shape,
                             interpolation=interpolation, copy=True
                            )
    return resampled

In [26]:
def process_patient(patient_dir: Path, output_base: Path, target_affine: np.ndarray,
                    target_shape: Tuple[int, int, int]) -> Dict[str, Path]:
    '''
    Process a single patient: resample all modalities + mask.
    Returns a dict of {modality: output_path}
    '''

    results = {}
    patient_id = patient_dir.name

    # I am going to choose a native T1 (t1n) as a reference and starting point
    t1n_path = get_NIfTI_path(ROOT_DIR, 't1n')

    ref_img = nib.load(t1n_path)
    ref_affine = ref_img.affine.copy()
    ref_shape = ref_img.shape[:3]

    # Resample all modalities using t1n's affine/shape as proxy for alignment
    for mod in MODALITIES:
        mod_path = get_NIfTI_path(ROOT_DIR, mod)

        # For clinical accuracy: resample to *same affine/shape as 1tn*
        # then optionally re-grid to target
        res = resample_image(mod_path, target_affine, target_shape, interpolation='continuous')
        out_path = output_base / patient_id / f'{mod}_resampled.nii.gz'
        nib.save(res, out_path)
        results[mod] = out_path

    # Resample tumor mask with nearest neighbor (discrete label)
    mask_path = get_NIfTI_path(ROOT_DIR, MASK_MODALITY)
    if mask_path.exists():
        mask_res = resample_image(mask_path, target_affine, target_shape, interpolation='nearest')
        out_mask_path = output_base / patient+id, f'{MASK_MODALITY}_resampled.nii.gz'
        nib.save(mask_res, out_mask_path)
        results[MASK_MODALITY] = out_mask_path

    return results

In [37]:
patients = get_patient_dirs(ROOT_DIR)
patients[1].name
#result = process_patient(ROOT_DIR, OUTPUT_ROOT, TARGET_AFFINE, TARGET_SHAPE)
#result = process_patient(patients, OUTPUT_ROOT, TARGET_AFFINE, TARGET_SHAPE)

'PatientID_0004'

----
<a name='do_work'></a>
## 2.0 <span style='color:blue'>|</span> Do the Work

In [27]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Find patients
patients = get_patient_dirs(ROOT_DIR)

process_fn = partial(process_patient, output_base=OUTPUT_ROOT, 
                     target_affine=TARGET_AFFINE, target_shape=TARGET_SHAPE)

results_list = Parallel(n_job=N_JOBS)(delayed(process_fn)(pid) for pid in patients)

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'list'